# Hybrid Quantization: MXFP4 + IQ4_NL + IQ4_XS

**Strategy:**
- `mxfp4` tensors (expert weights) → `copy` (keep as-is)
- 2880-column tensors (not divisible by 256) → `iq4_nl` (32-block, no fallback)
- everything else → `IQ4_XS` + imatrix (best quality for attention/embed)

**Steps:**
1. Discover mxfp4 tensors and 2880-column fallback tensors
2. Export calibration data from SFT training dataset
3. Generate imatrix
4. Run custom quantization

In [6]:
import os, re, subprocess

MXFP4_GGUF    = "gpt-oss-20b.MXFP4.gguf"
LLAMA_DIR     = "llama.cpp"
QUANTIZER     = f"{LLAMA_DIR}/llama-quantize"
IMATRIX_BIN   = f"{LLAMA_DIR}/llama-imatrix"
CALIB_FILE    = "calibration.txt"
IMATRIX_FILE  = "gpt-oss-20b-imatrix.dat"
OUTPUT_GGUF   = "gpt-oss-20b-sft-IQ4_XS-hybrid.gguf"
LD_PATH       = f"{LLAMA_DIR}:{os.environ.get('LD_LIBRARY_PATH', '')}"
ENV           = {**os.environ, "LD_LIBRARY_PATH": LD_PATH}

for f in [MXFP4_GGUF, QUANTIZER, IMATRIX_BIN]:
    status = "✅" if os.path.exists(f) else "❌"
    print(f"{status} {f}")

✅ gpt-oss-20b.MXFP4.gguf
✅ llama.cpp/llama-quantize
✅ llama.cpp/llama-imatrix


## Step 1 — Discover MXFP4 tensors and 2880-column fallback tensors

Run this in the terminal first, then come back to parse the log:

```bash
LD_LIBRARY_PATH=llama.cpp:$LD_LIBRARY_PATH llama.cpp/llama-quantize \
  --allow-requantize \
  gpt-oss-20b.MXFP4.gguf \
  /tmp/dry.gguf \
  IQ4_XS 2>&1 | tee quantize_dry_run.log

rm /tmp/dry.gguf
```

In [7]:
LOG_FILE = "quantize_dry_run.log"

if not os.path.exists(LOG_FILE):
    raise FileNotFoundError(
        f"{LOG_FILE} not found.\n"
        "Run the terminal command in the markdown cell above first."
    )

with open(LOG_FILE) as f:
    output = f.read()

print(f"Log file: {len(output.splitlines())} lines")

# Parse mxfp4 tensors
mxfp4_tensors = set()
for line in output.splitlines():
    if "mxfp4" in line and "]" in line:
        m = re.search(r"\]\s+(\S+)\s+-", line)
        if m:
            generic = re.sub(r"blk\.\d+\.", "blk.*.", m.group(1))
            mxfp4_tensors.add(generic)

# Parse fallback tensors (ncols not divisible by 256)
fallback_tensors = set()
for line in output.splitlines():
    if "not divisible" in line or "falling back" in line:
        m = re.search(r"warning:\s+(\S+)\s+-", line)
        if m:
            generic = re.sub(r"blk\.\d+\.", "blk.*.", m.group(1))
            fallback_tensors.add(generic)

fallback_tensors -= mxfp4_tensors

print(f"\nMXFP4 tensors (will COPY): {len(mxfp4_tensors)}")
for t in sorted(mxfp4_tensors): print(f"  {t}")

print(f"\nFallback tensors (will use IQ4_NL): {len(fallback_tensors)}")
for t in sorted(fallback_tensors): print(f"  {t}")

Log file: 1528 lines

MXFP4 tensors (will COPY): 3
  blk.*.ffn_down_exps.weight
  blk.*.ffn_gate_exps.weight
  blk.*.ffn_up_exps.weight

Fallback tensors (will use IQ4_NL): 4
  blk.*.attn_k.weight
  blk.*.attn_q.weight
  blk.*.attn_v.weight
  token_embd.weight


## Step 2 — Calibration data from SFT training dataset

In [8]:
import json, random
from collections import defaultdict

SFT_JSON      = "train_sft_final.json"
CALIB_SAMPLES = 512   # total samples

if os.path.exists(CALIB_FILE):
    print(f"✅ {CALIB_FILE} already exists, skipping.")
else:
    if not os.path.exists(SFT_JSON):
        raise FileNotFoundError(
            f"{SFT_JSON} not found. Run SFT_datasets.ipynb first."
        )

    # Load — supports both JSON array and JSONL (one object per line)
    with open(SFT_JSON) as f:
        first_char = f.read(1)
        f.seek(0)
        if first_char == "[":
            data = json.load(f)
        else:
            data = [json.loads(line) for line in f if line.strip()]

    print(f"Loaded {len(data)} entries from {SFT_JSON}")

    # Detect text field
    sample = data[0]
    print(f"Fields: {list(sample.keys())}")
    text_col = next((k for k in ["text", "content", "instruction"] if k in sample), None)
    if text_col is None:
        raise ValueError(f"No text field found. Available: {list(sample.keys())}")
    print(f"Using text field: '{text_col}'")

    # Group by source
    by_source = defaultdict(list)
    for row in data:
        by_source[row.get("source", "unknown")].append(row)

    total = len(data)

    # Proportional sampling — mirrors actual training distribution
    print(f"\nProportional sampling ({CALIB_SAMPLES} total):")
    selected = []
    for src in sorted(by_source.keys()):
        pool = by_source[src]
        proportion = len(pool) / total
        n = max(1, round(CALIB_SAMPLES * proportion))
        n = min(n, len(pool))
        sampled = random.sample(pool, n)
        selected.extend(sampled)
        print(f"  {src:<20} {proportion*100:5.1f}%  →  {n} samples")

    random.shuffle(selected)
    print(f"\nTotal: {len(selected)} samples")

    with open(CALIB_FILE, "w") as f:
        for row in selected:
            text = row[text_col].strip()
            if text:
                f.write(text + "\n")

    print(f"✅ Written to {CALIB_FILE}")

# Verify
with open(CALIB_FILE) as f:
    lines = f.readlines()
print(f"Calibration file: {len(lines)} lines, {os.path.getsize(CALIB_FILE)/1e6:.1f} MB")

Loaded 30118 entries from train_sft_final.json
Fields: ['instruction', 'response', 'source']
Using text field: 'instruction'

Proportional sampling (512 total):
  GSM8K                 24.8%  →  127 samples
  OpenThoughts          34.6%  →  177 samples
  alpaca                 5.1%  →  26 samples
  arc                    2.2%  →  11 samples
  boolq                  5.6%  →  29 samples
  dolly                  4.3%  →  22 samples
  lmsys_conv             5.1%  →  26 samples
  math                   2.5%  →  13 samples
  sharegpt4             11.9%  →  61 samples
  ultrachat              4.0%  →  20 samples

Total: 512 samples
✅ Written to calibration.txt
Calibration file: 2633 lines, 0.2 MB


## Step 3 — Generate imatrix (~20 min on A100)

In [9]:
if os.path.exists(IMATRIX_FILE):
    print(f"✅ {IMATRIX_FILE} already exists, skipping.")
else:
    cmd = [
        IMATRIX_BIN,
        "-m", MXFP4_GGUF,
        "-f", CALIB_FILE,
        "-o", IMATRIX_FILE,
        "-ngl", "99",
        "--chunks", "128",
    ]
    print("Running:", " ".join(cmd))
    subprocess.run(cmd, check=True, env=ENV)
    print(f"\n✅ {IMATRIX_FILE} ready ({os.path.getsize(IMATRIX_FILE)/1e6:.1f} MB)")

Running: llama.cpp/llama-imatrix -m gpt-oss-20b.MXFP4.gguf -f calibration.txt -o gpt-oss-20b-imatrix.dat -ngl 99 --chunks 128


ggml_cuda_init: found 1 CUDA devices (Total VRAM: 81151 MiB):
  Device 0: NVIDIA A100-SXM4-80GB, compute capability 8.0, VMM: yes, VRAM: 81151 MiB
common_init_result: fitting params to device memory, for bugs during this step try to reproduce them with -fit off, or provide --verbose logs if the bug only occurs with -fit on
llama_params_fit_impl: projected to use 12531 MiB of device memory vs. 80723 MiB of free device memory
llama_params_fit_impl: will leave 68192 >= 1024 MiB of free device memory, no changes needed
llama_params_fit: successfully fit params to free device memory
llama_params_fit: fitting params to free memory took 1.40 seconds
llama_model_load_from_file_impl: using device CUDA0 (NVIDIA A100-SXM4-80GB) (0000:00:05.0) - 80723 MiB free
llama_model_loader: loaded meta data with 36 key-value pairs and 459 tensors from gpt-oss-20b.MXFP4.gguf (version GGUF V3 (latest))
llama_model_loader: Dumping metadata keys/values. Note: KV overrides do not apply in this output.
llama_model

3.10 minutes
[1]320.1881,[2]319.8579,[3]482.0194,[4]305.4817,[5]276.3891,[6]267.5662,[7]244.4361,[8]231.7923,


save_imatrix: saving imatrix using GGUF format with a different suffix than .gguf
save_imatrix: if you want the previous imatrix format, use --output-format dat



[9]233.6543,[10]265.2907,[11]260.7195,[12]276.9994,[13]271.3300,[14]263.7258,[15]223.3448,[16]210.2790,

...


[89]231.1634,[90]243.9283,[91]246.5851,[92]243.8949,[93]243.8535,[94]244.8849,[95]242.0996,[96]243.5259,


save_imatrix: saving imatrix using GGUF format with a different suffix than .gguf
save_imatrix: if you want the previous imatrix format, use --output-format dat



[97]242.8478,[98]244.1456,[99]243.0511,[100]244.0313,[101]245.1517,
Final estimate: PPL = 245.1517 +/- 5.31568





save_imatrix: saving imatrix using GGUF format with a different suffix than .gguf
save_imatrix: if you want the previous imatrix format, use --output-format dat
llama_perf_context_print:        load time =    5149.20 ms
llama_perf_context_print: prompt eval time =  121170.55 ms / 51712 tokens (    2.34 ms per token,   426.77 tokens per second)
llama_perf_context_print:        eval time =       0.00 ms /     1 runs   (    0.00 ms per token,      inf tokens per second)
llama_perf_context_print:       total time =  141816.29 ms / 51713 tokens
llama_perf_context_print:    graphs reused =         50



✅ gpt-oss-20b-imatrix.dat ready (28.1 MB)


## Step 4 — Hybrid quantization: MXFP4 copy + IQ4_NL + IQ4_XS

In [13]:
# Build --tensor-type args dynamically from Step 1 discoveries

tensor_type_args = []

# MXFP4 expert weights → copy (keep as-is)
for t in sorted(mxfp4_tensors):
    tensor_type_args += ["--tensor-type", f"{t}=iq4_nl"]

# 2880-column fallback tensors → iq4_nl (32-block, no fallback)
for t in sorted(fallback_tensors):
    tensor_type_args += ["--tensor-type", f"{t}=iq4_nl"]

print("Tensor type overrides:")
for i in range(0, len(tensor_type_args), 2):
    print(f"  {tensor_type_args[i+1]}")

cmd = [
    QUANTIZER,
    "--allow-requantize",
    "--imatrix", IMATRIX_FILE,
] + tensor_type_args + [
    MXFP4_GGUF,
    OUTPUT_GGUF,
    "IQ4_XS",
]

QUANT_LOG = "quantize_final.log"
print(f"\nOutput: {OUTPUT_GGUF}")
print(f"Log:    {QUANT_LOG}")
print("Running quantization ...\n")

result = subprocess.run(cmd, capture_output=True, text=True, env=ENV)
with open(QUANT_LOG, "w") as f:
    f.write(result.stdout + result.stderr)

if result.returncode != 0:
    print("ERROR — last 20 lines of log:")
    for line in (result.stdout + result.stderr).splitlines()[-20:]:
        print(f"  {line}")
    raise RuntimeError("Quantization failed")

size_gb = os.path.getsize(OUTPUT_GGUF) / 1e9
print(f"✅ Done: {OUTPUT_GGUF} — {size_gb:.1f} GB")
print(f"✅ Log saved: {QUANT_LOG}")

Tensor type overrides:
  blk.*.ffn_down_exps.weight=iq4_nl
  blk.*.ffn_gate_exps.weight=iq4_nl
  blk.*.ffn_up_exps.weight=iq4_nl
  blk.*.attn_k.weight=iq4_nl
  blk.*.attn_q.weight=iq4_nl
  blk.*.attn_v.weight=iq4_nl
  token_embd.weight=iq4_nl

Output: gpt-oss-20b-sft-IQ4_XS-hybrid.gguf
Log:    quantize_final.log
Running quantization ...

✅ Done: gpt-oss-20b-sft-IQ4_XS-hybrid.gguf — 12.1 GB
✅ Log saved: quantize_final.log


In [14]:
# Verify: parse quantize_final.log and show what each tensor was actually quantized to.
# f32 tensors show no "converting to" — confirms they were left untouched.

from collections import Counter

with open(QUANT_LOG) as f:
    log = f.read()

final_types = Counter()   # what tensors ended up as
kept_f32    = []          # tensors kept at f32 (not quantized)
kept_mxfp4  = []          # tensors kept at mxfp4 (copy)
fallback_used = []        # tensors that still fell back unexpectedly

for line in log.splitlines():
    if not line.startswith("["):
        continue

    # Quantized tensor: "converting to X"
    m = re.search(r"converting to\s+(\S+)", line)
    if m:
        final_types[m.group(1)] += 1
        continue

    # Kept as-is (f32 / copy): line has "type = X" but no "converting to"
    m_name = re.search(r"\]\s+(\S+)\s+-", line)
    m_type = re.search(r"type\s*=\s*(\S+)", line)
    if m_name and m_type:
        t = m_type.group(1)
        name = m_name.group(1)
        final_types[t] += 1
        if t == "f32":
            kept_f32.append(name)
        elif t == "mxfp4":
            kept_mxfp4.append(name)

print("=== Final tensor type summary ===")
for t, count in sorted(final_types.items(), key=lambda x: -x[1]):
    print(f"  {t:<12} {count:>4} tensors")

print(f"\n✅ f32 tensors kept untouched: {len(kept_f32)}")
for n in kept_f32[:5]: print(f"  {n}")
if len(kept_f32) > 5: print(f"  ... and {len(kept_f32)-5} more")

print(f"\n✅ mxfp4 tensors copied as-is: {len(kept_mxfp4)}")
for n in kept_mxfp4[:5]: print(f"  {n}")
if len(kept_mxfp4) > 5: print(f"  ... and {len(kept_mxfp4)-5} more")

=== Final tensor type summary ===
  f32,          289 tensors
  iq4_nl        144 tensors
  iq4_xs         24 tensors
  bf16,           2 tensors

✅ f32 tensors kept untouched: 0

✅ mxfp4 tensors copied as-is: 0


In [ ]:
# Quick inference test
cmd = [
    f"{LLAMA_DIR}/llama-cli",
    "--model", OUTPUT_GGUF,
    "--n-gpu-layers", "99",
    "-p", "Explain step by step why the sky is blue.",
    "-n", "256",
    "--log-disable",
]
result = subprocess.run(cmd, capture_output=True, text=True, env=ENV)
print(result.stdout[-3000:] if result.stdout else result.stderr[-1000:])